# Module 2 — Build Your First AI Agent from Scratch

**Course:** Build Your First AI Agent from Scratch  
**Community:** [Autonomous MSP](https://skool.com/autonomous-msp-2162)  

This notebook walks through every step of Module 2:  
- The ReAct pattern (Reason + Act loop)  
- The TOOLS registry  
- The response parser  
- The main agent loop  
- Running your first OSPF diagnosis  

**Run each cell in order.** No prior setup required beyond your Anthropic API key.

## Step 1 — Install the Anthropic SDK

In [ ]:
!pip install anthropic -q

## Step 2 — Set Your API Key

**Option A (recommended for Colab):** Add your key to Colab Secrets.
- Click the key icon in the left sidebar (or go to Tools → Secrets)
- Add a secret named `ANTHROPIC_API_KEY` with your key value
- Enable notebook access for that secret

**Option B:** Paste your key directly into the cell below (don't share or commit this notebook if you do).

In [ ]:
import os

# Option A — read from Colab Secrets (recommended)
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("API key loaded from Colab Secrets.")
except Exception:
    # Option B — paste directly (local Jupyter or if Secrets not set up)
    os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."  # <-- replace with your key
    print("API key set directly. Do not share this notebook.")

## Step 3 — Demo Tools

These are **simulated** network tools — hardcoded return values that mimic what a real router would respond with.  
In Module 3 you replace these with real Netmiko SSH calls.

Right now they give us a working agent loop we can reason about without needing lab access.

In [ ]:
def show_ospf_neighbors(device: str) -> str:
    data = {
        "R1": "Gi0/1: neighbor 10.0.0.2  State INIT  Dead 34\n"
              "Gi0/2: neighbor 10.0.0.6  State FULL  Dead 38",
        "R2": "Gi0/0: neighbor 10.0.0.1  State INIT  Dead 31",
    }
    return data.get(device, f"Device {device} not found")


def show_ospf_interface(device: str, interface: str) -> str:
    data = {
        ("R1", "Gi0/1"): "Area 0, Hello 10, Dead 40, MTU 1500, auth: none",
        ("R2", "Gi0/0"): "Area 1, Hello 10, Dead 40, MTU 1500, auth: none",
    }
    return data.get((device, interface), f"Interface {interface} not found on {device}")


def ping_device(device: str, target: str) -> str:
    return f"{device} -> {target}: Success rate 0 percent (0/5)"


print("Tools defined.")
print("Test show_ospf_neighbors('R1'):", show_ospf_neighbors("R1"))

## Step 4 — The TOOLS Registry

The TOOLS dict is the agent's toolbox. Each entry has:
- `function` — the Python callable
- `description` — what the LLM reads to decide which tool to call
- `params` — what parameters the tool expects

The description and params get injected into the system prompt automatically.  
**Adding a new tool = adding one entry here. Nothing else changes.**

In [ ]:
TOOLS = {
    "show_ospf_neighbors": {
        "function": show_ospf_neighbors,
        "description": "Get OSPF neighbor table from a router",
        "params": {"device": "Device hostname (string)"},
    },
    "show_ospf_interface": {
        "function": show_ospf_interface,
        "description": "Get OSPF config for a specific interface",
        "params": {"device": "Device hostname", "interface": "Interface name"},
    },
    "ping_device": {
        "function": ping_device,
        "description": "Ping a target IP from a source device",
        "params": {"device": "Source device hostname", "target": "Target IP address"},
    },
}

print(f"Registered {len(TOOLS)} tools:", list(TOOLS.keys()))

## Step 5 — The System Prompt

This is the **contract** between you and the LLM.  
It tells the model exactly what format to output so your Python parser can read it reliably.

Without this, you get prose — useful for humans, useless for loops.

In [ ]:
SYSTEM_PROMPT = """You are a network troubleshooting agent for an MSP.

For every problem, follow this exact format:

Thought: [your reasoning about what to check next]
Action: [tool_name]
Params: {{"param": "value"}}

OR when you have a diagnosis:

Final Answer: [root cause and recommended fix]

Available tools:
{tools_description}

Rules:
- Always start with a Thought.
- Never give a Final Answer before using at least one tool.
- Be specific and actionable.
"""

print("System prompt ready.")

## Step 6 — The Response Parser

The LLM outputs text. The parser extracts:
- `Final Answer:` → stop the loop, return the diagnosis
- `Action:` + `Params:` → run that tool, feed the result back
- Neither → return an error

In [ ]:
import re
import json


def parse_response(text: str) -> dict:
    if "Final Answer:" in text:
        return {"type": "final", "answer": text.split("Final Answer:")[-1].strip()}

    tool_match = re.search(r"Action:\s*(\w+)", text)
    params_match = re.search(r"Params:\s*(\{.*?\})", text, re.DOTALL)

    if not tool_match:
        return {"type": "error", "message": "No action found"}

    params = json.loads(params_match.group(1)) if params_match else {}
    return {"type": "action", "tool": tool_match.group(1).strip(), "params": params}


# Quick test
test_response = 'Thought: checking neighbor table\nAction: show_ospf_neighbors\nParams: {"device": "R1"}'
print("Parser test:", parse_response(test_response))

## Step 7 — The Main Agent Loop

This is the engine. Every iteration:
1. Send the full conversation history to the LLM
2. Parse the response
3. If `Final Answer` — stop and return
4. If `Action` — run the tool, append the `Observation` to the conversation
5. Repeat

`max_iterations` is your safety valve against infinite loops.

In [ ]:
import anthropic

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from environment


def run_agent(problem: str, max_iterations: int = 8) -> str:
    tools_desc = "\n".join(
        f"- {n}: {i['description']} | params: {i['params']}"
        for n, i in TOOLS.items()
    )
    messages = [{"role": "user", "content": problem}]
    print(f"\n[Agent] Problem: {problem}\n{'─'*60}")

    for iteration in range(max_iterations):
        print(f"\n[Iteration {iteration + 1}]")

        response = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=1024,
            system=SYSTEM_PROMPT.format(tools_description=tools_desc),
            messages=messages,
        )
        raw = response.content[0].text
        print(raw)
        messages.append({"role": "assistant", "content": raw})

        parsed = parse_response(raw)

        if parsed["type"] == "final":
            print(f"\n{'─'*60}\n[Agent] Done.\n")
            return parsed["answer"]

        if parsed["type"] == "error" or parsed["tool"] not in TOOLS:
            return f"Agent error: {parsed.get('message', 'unknown tool')}"

        result = TOOLS[parsed["tool"]]["function"](**parsed["params"])
        print(f"\n[Tool Result] {result}")
        messages.append({"role": "user", "content": f"Observation: {result}"})

    return "Max iterations reached."


print("Agent loop ready.")

## Step 8 — Run Your First Diagnosis

This is the module challenge. Run the cell below and watch the agent loop through the OSPF INIT problem step by step.

Expected: 4 iterations — check R1 neighbors, check R1 interface, check R2 interface, final answer (area mismatch).

In [ ]:
answer = run_agent(
    "R1 is showing an OSPF neighbor stuck in INIT state. "
    "Diagnose the root cause and recommend a fix."
)

print("\n[Final Answer]")
print(answer)

## Exercise — Try a Different Problem

Modify the problem string below and re-run. The agent will use whatever tools and reasoning it needs.

Notice: it may hit a tool that returns `not found` — that is expected. Watch how it handles missing data and whether it still reaches a conclusion.

In [ ]:
answer = run_agent(
    "R2 is showing an OSPF neighbor stuck in INIT state. "
    "Diagnose the root cause and recommend a fix."
)

print("\n[Final Answer]")
print(answer)

## What's Next

You now have a working ReAct agent in ~80 lines of Python. You understand every line.

**Module 3** replaces the demo tool functions with real Netmiko SSH calls to actual devices — same loop, real data.

---

**Module Challenge:** Post a screenshot of your [Iteration 1–4] + [Final Answer] output in **#agent-builds** in the community.  
First member to share gets a shoutout in the next live session.